# 模块四：三模块标准化接口与 scVI-FBA 耦合原型

本 Notebook 将三个独立模块通过明确的数学接口连接起来，重点实现 scVI -> FBA 的数据驱动约束接口，
并给出 HH <-> FBA 的双向接口原型。针对线性接口 R2 的解释性陷阱，补充非线性映射对比。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.interpolate import interp1d
import cobra
from cobra.io import load_model
import scanpy as sc
import scvi
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120

## 1. 接口总览

| 方向 | 输出量 | 输入量 | 数学形式 |
|------|--------|--------|----------|
| HH -> FBA | ATP 消耗速率 | FBA 边界条件 | v_ATPase >= ATP_demand |
| FBA -> HH | 代谢物浓度 | HH 扩展参数 | g_KATP = f([ATP]/[ADP]) |
| scVI -> FBA | 潜在空间 z | 酶表达量 E | v_max,j = E_j * kcat,j / conversion |
| scVI -> HH | 细胞状态 | 电导参数 | g_Na, g_K ~ N(mu(z), sigma(z)) |

## 2. HH -> FBA：ATP 需求边界条件

In [ ]:
def hh_to_atp_demand(firing_rate_hz, atp_per_spike=1e-9, membrane_area_cm2=1.0):
    atp_demand_mol_per_s = firing_rate_hz * atp_per_spike
    gdw = membrane_area_cm2 * 1e-12
    atp_demand = atp_demand_mol_per_s * 1000 * 3600 / gdw
    return atp_demand

for fr in [0, 1, 5, 10, 20]:
    demand = hh_to_atp_demand(fr)
    print(f"发放频率 {fr} Hz -> ATP demand: {demand:.2e} mmol/gDW/h")

model = load_model('e_coli_core')
model.reactions.EX_glc__D_e.lower_bound = -10
model.reactions.EX_o2_e.lower_bound = -20

atpm = model.reactions.ATPM
atp_demands = np.linspace(0, 20, 21)
growth_rates = []
for d in atp_demands:
    with model:
        atpm.lower_bound = d
        sol = model.optimize()
        growth_rates.append(sol.objective_value if sol.status == 'optimal' else 0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(atp_demands, growth_rates, 'o-', color='darkgreen')
ax.set_xlabel('ATP maintenance demand (mmol/gDW/h)')
ax.set_ylabel('Max growth rate (1/h)')
ax.set_title('HH -> FBA: ATP Demand Penalty on Growth')
ax.grid(True)
plt.tight_layout()
plt.savefig('report_images/interface_hh_to_fba_atp.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. FBA -> HH：代谢物浓度调节 K-ATP 通道

In [ ]:
def hh_with_katp(y, t, I_ext, atp_adp_ratio=10.0, g_KATP_max=0.5,
                 g_Na=120.0, g_K=36.0, g_L=0.3, C_m=1.0):
    V, m, h, n = y
    alpha_m = 0.1 * (V + 40) / (1 - np.exp(-(V + 40)/10))
    beta_m  = 4.0 * np.exp(-(V + 65)/18)
    alpha_h = 0.07 * np.exp(-(V + 65)/20)
    beta_h  = 1.0 / (1 + np.exp(-(V + 35)/10))
    alpha_n = 0.01 * (V + 55) / (1 - np.exp(-(V + 55)/10))
    beta_n  = 0.125 * np.exp(-(V + 65)/80)

    E_Na, E_K, E_L = 50.0, -77.0, -54.387
    g_KATP = g_KATP_max / (1 + (atp_adp_ratio / 2.0)**2)

    I_Na = g_Na * m**3 * h * (V - E_Na)
    I_K  = g_K * n**4 * (V - E_K)
    I_L  = g_L * (V - E_L)
    I_KATP = g_KATP * (V - E_K)

    dVdt = (I_ext - (I_Na + I_K + I_L + I_KATP)) / C_m
    dmdt = alpha_m * (1 - m) - beta_m * m
    dhdt = alpha_h * (1 - h) - beta_h * h
    dndt = alpha_n * (1 - n) - beta_n * n
    return [dVdt, dmdt, dhdt, dndt]

def simulate_katp(atp_adp_ratio, I_amp=10.0, tmax=100):
    time = np.linspace(0, tmax, int(tmax*100)+1)
    I_ext = np.zeros_like(time)
    I_ext[(time > 10) & (time < 50)] = I_amp
    I_func = interp1d(time, I_ext, fill_value='extrapolate')
    y0 = [-65.0, 0.05, 0.6, 0.32]
    sol = odeint(lambda y, t: hh_with_katp(y, t, I_func(t), atp_adp_ratio),
                 y0, time, hmax=0.1)
    return time, sol[:, 0]

ratios = [1, 5, 10, 50]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
axes = axes.flatten()
for ax, r in zip(axes, ratios):
    t, V = simulate_katp(r)
    ax.plot(t, V, color='navy')
    ax.set_title(f'ATP/ADP = {r}, g_KATP = {0.5/(1+(r/2)**2):.3f}')
    ax.set_ylabel('V (mV)')
    ax.grid(True)
for ax in axes[-2:]:
    ax.set_xlabel('Time (ms)')
plt.tight_layout()
plt.savefig('report_images/interface_fba_to_hh_katp.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. scVI -> FBA：潜在空间映射到酶约束（线性 + 非线性对比）

核心接口：
1. scVI 学习潜在空间 z
2. 映射 z -> E：比较线性回归、随机森林、MLP
3. 酶约束：v_max = E * kcat / conversion
4. 上下文特异性 FBA

若线性 R2 已 > 0.7，说明潜在空间捕获了代谢相关变异；若 < 0.5，需非线性接口或调整潜在空间维度。

In [ ]:
adata = sc.datasets.pbmc3k()
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=1000)
adata = adata[:, adata.var['highly_variable']].copy()
if hasattr(adata.X, 'toarray'):
    adata.X = adata.X.toarray()
adata.X = np.expm1(adata.X).round().astype(np.float32)

scvi.model.SCVI.setup_anndata(adata)
scvi_model = scvi.model.SCVI(adata, n_latent=10, n_hidden=128)
scvi_model.train(max_epochs=20, accelerator='cpu', plan_kwargs={'lr': 1e-3})
z = scvi_model.get_latent_representation(adata)

# 模拟酶表达
np.random.seed(42)
target_reactions = ['GLCpts', 'PFK', 'PGK', 'PYK', 'PDH', 'CS', 'ICDHyr', 'SUCDi']
n_cells = adata.n_obs
n_reactions = len(target_reactions)
W = np.random.randn(z.shape[1], n_reactions) * 0.1
b = np.random.uniform(0.5, 2.0, n_reactions)
enzyme_expression = z @ W + b
enzyme_expression = np.maximum(enzyme_expression, 0.1)

# 线性与非线性映射对比
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_train, X_test, y_train, y_test = train_test_split(z, enzyme_expression, test_size=0.2, random_state=42)

models = {
    'Linear': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    'MLP': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
}

r2_results = {name: [] for name in models}
for name, m in models.items():
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    for j in range(n_reactions):
        r2_results[name].append(r2_score(y_test[:, j], y_pred[:, j]))

x = np.arange(n_reactions)
width = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, r2s) in enumerate(r2_results.items()):
    ax.bar(x + i*width, r2s, width, label=name)
ax.axhline(0.7, color='green', linestyle='--', alpha=0.5, label='R2=0.7 threshold')
ax.set_ylabel('R2 (z -> enzyme expression)')
ax.set_title('Linear vs Nonlinear scVI -> FBA Mapping')
ax.set_xticks(x + width)
ax.set_xticklabels(target_reactions, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.savefig('report_images/interface_scvi_fba_r2.png', dpi=150, bbox_inches='tight')
plt.show()

print("各映射方法平均 R2:")
for name, r2s in r2_results.items():
    print(f"  {name}: {np.mean(r2s):.3f}")

In [ ]:
# 用最佳映射方法（这里选线性作为默认）做上下文特异性 FBA
best_model = models['Linear']
best_model.fit(z, enzyme_expression)
predicted_enzyme = best_model.predict(z)

cell_scores = predicted_enzyme.sum(axis=1)
low_idx = np.argmin(cell_scores)
high_idx = np.argmax(cell_scores)

print(f"低代谢活性细胞 index: {low_idx}, score: {cell_scores[low_idx]:.2f}")
print(f"高代谢活性细胞 index: {high_idx}, score: {cell_scores[high_idx]:.2f}")

fba_model = load_model('e_coli_core')
fba_model.reactions.EX_glc__D_e.lower_bound = -10
fba_model.reactions.EX_o2_e.lower_bound = -20

kcat_values = {'GLCpts': 80, 'PFK': 120, 'PGK': 200, 'PYK': 150,
               'PDH': 60, 'CS': 100, 'ICDHyr': 90, 'SUCDi': 70}
conversion = 3600.0

def context_specific_fba(model, enzyme_vec, rxn_list, kcat_dict):
    with model:
        for j, rxn_id in enumerate(rxn_list):
            if rxn_id not in [r.id for r in model.reactions]:
                continue
            rxn = model.reactions.get_by_id(rxn_id)
            vmax = enzyme_vec[j] * kcat_dict.get(rxn_id, 50) / conversion
            if rxn.upper_bound > 0:
                rxn.upper_bound = min(rxn.upper_bound, vmax)
        sol = model.optimize()
        return sol

sol_low = context_specific_fba(fba_model, predicted_enzyme[low_idx], target_reactions, kcat_values)
sol_high = context_specific_fba(fba_model, predicted_enzyme[high_idx], target_reactions, kcat_values)

print(f"\n低代谢活性细胞生长速率: {sol_low.objective_value:.4f} /h")
print(f"高代谢活性细胞生长速率: {sol_high.objective_value:.4f} /h")

flux_low = [sol_low.fluxes[r] for r in target_reactions]
flux_high = [sol_high.fluxes[r] for r in target_reactions]
x = np.arange(len(target_reactions))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width/2, flux_low, width, label='Low metabolic activity cell', color='steelblue')
ax.bar(x + width/2, flux_high, width, label='High metabolic activity cell', color='coral')
ax.set_ylabel('Flux (mmol/gDW/h)')
ax.set_title('scVI -> FBA: Context-Specific Flux Predictions')
ax.set_xticks(x)
ax.set_xticklabels(target_reactions, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.savefig('report_images/interface_scvi_to_fba.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 小节：三模块接口落地清单

| 接口 | 数学形式 | 已实现 | 下一步 |
|------|----------|--------|--------|
| HH -> FBA | v_ATPM >= f(firing_rate) | 是 | 标定 atp_per_spike 参数 |
| FBA -> HH | g_KATP = f(ATP/ADP) | 是 | 用真实代谢物浓度校准 |
| scVI -> FBA | v_max = E(z) * kcat | 是（线性 + 非线性对比） | 用真实单细胞蛋白质组数据替换模拟 E |
| scVI -> HH | g_Na/g_K ~ z | 部分 | 需电生理 + 转录组联合数据集 |